In [0]:
from pyspark.sql.functions import when, col

flights_path = "/Volumes/mini_cap/default/airline_data/flights.csv"
bookings_path = "/Volumes/mini_cap/default/airline_data/bookings.csv"

df_flights = spark.read.option("header", "true").option("inferSchema", "true").csv(flights_path)
df_bookings = spark.read.option("header", "true").option("inferSchema", "true").csv(bookings_path)

In [0]:
df_bookings_transformed = df_bookings.withColumn(
    "price_band",
    when(col("ticket_price") > 20000, "Premium")
    .when(col("ticket_price") > 10000, "Standard")
    .otherwise("Budget")
)

display(df_bookings_transformed)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,price_band
B1001,F101,Rahul Sharma,Economy,8500,2026-06-01,Budget
B1002,F101,Priya Reddy,Business,22000,2026-06-01,Premium
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,Budget
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,Standard
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,Budget
B1006,F105,Neha Singh,Business,25000,2026-06-03,Premium
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,Budget
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,Standard
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,Budget
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,Premium


In [0]:
df_flights_transformed = df_flights.withColumn(
    "delay_flag",
    when(col("status") == "Delayed", "Yes").otherwise("No")
)

display(df_flights_transformed)

flight_id,airline,from_city,to_city,duration,status,delay_flag
F101,Indigo,Hyderabad,Delhi,140,On Time,No
F102,Air India,Mumbai,Chennai,120,Delayed,Yes
F103,Vistara,Bangalore,Hyderabad,90,On Time,No
F104,Indigo,Delhi,Mumbai,130,Cancelled,No
F105,Air India,Chennai,Bangalore,80,On Time,No
F106,Akasa,Pune,Delhi,150,Delayed,Yes
F107,Vistara,Hyderabad,Kolkata,160,On Time,No
F108,Indigo,Mumbai,Hyderabad,110,On Time,No
F109,Akasa,Delhi,Chennai,145,Delayed,Yes
F110,Air India,Bangalore,Mumbai,95,On Time,No


In [0]:
preferences_path = "/Volumes/mini_cap/default/airline_data/Preferences.json"

df_preferences = (
    spark.read
    .option("multiLine", "true")
    .json(preferences_path)
    .select("passenger_name", "meal", "seat", "extra_baggage")
)
df_journey = (
    df_bookings_transformed
    .join(df_flights_transformed, on="flight_id", how="inner")
    .join(df_preferences, on="passenger_name", how="left")
)

display(df_journey)

passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat,extra_baggage
Rahul Sharma,F101,B1001,Economy,8500,2026-06-01,Budget,Indigo,Hyderabad,Delhi,140,On Time,No,Veg,Window,true
Priya Reddy,F101,B1002,Business,22000,2026-06-01,Premium,Indigo,Hyderabad,Delhi,140,On Time,No,Non-Veg,Aisle,false
Amit Kumar,F102,B1003,Economy,9000,2026-06-02,Budget,Air India,Mumbai,Chennai,120,Delayed,Yes,Veg,Middle,false
Sneha Patel,F103,B1004,Premium Economy,15000,2026-06-02,Standard,Vistara,Bangalore,Hyderabad,90,On Time,No,Jain,Window,true
Farhan Ali,F104,B1005,Economy,7500,2026-06-03,Budget,Indigo,Delhi,Mumbai,130,Cancelled,No,Non-Veg,Aisle,false
Neha Singh,F105,B1006,Business,25000,2026-06-03,Premium,Air India,Chennai,Bangalore,80,On Time,No,Veg,Window,true
Arjun Verma,F106,B1007,Economy,10000,2026-06-04,Budget,Akasa,Pune,Delhi,150,Delayed,Yes,Veg,Middle,false
Meera Nair,F107,B1008,Premium Economy,17000,2026-06-04,Standard,Vistara,Hyderabad,Kolkata,160,On Time,No,Jain,Window,true
Kiran Rao,F108,B1009,Economy,9500,2026-06-05,Budget,Indigo,Mumbai,Hyderabad,110,On Time,No,Veg,Aisle,false
Nisha Reddy,F109,B1010,Business,28000,2026-06-05,Premium,Akasa,Delhi,Chennai,145,Delayed,Yes,Non-Veg,Window,true


In [0]:
df_journey.filter(col("meal").isNull()).select("passenger_name", "meal", "seat").display()

passenger_name,meal,seat
Vikram Singh,null,null
Anjali Rao,null,null
Faiz Ahmed,null,null
Megha Kapoor,null,null


In [0]:
df_journey.createOrReplaceTempView("passenger_journey")

In [0]:
spark.sql("""
    SELECT airline, SUM(ticket_price) AS total_revenue
    FROM passenger_journey
    GROUP BY airline
    ORDER BY total_revenue DESC
""").display()

airline,total_revenue
Indigo,90000
Vistara,71500
Air India,68000
Akasa,62000


In [0]:
spark.sql("""
    SELECT from_city, to_city, SUM(ticket_price) AS total_revenue
    FROM passenger_journey
    GROUP BY from_city, to_city
    ORDER BY total_revenue DESC
""").display()

from_city,to_city,total_revenue
Hyderabad,Delhi,39000
Bangalore,Hyderabad,38000
Delhi,Chennai,28000
Hyderabad,Kolkata,26500
Chennai,Bangalore,25000
Chennai,Pune,24000
Bangalore,Mumbai,23500
Delhi,Hyderabad,18000
Hyderabad,Goa,16000
Kolkata,Bangalore,10500


In [0]:
spark.sql("""
    SELECT AVG(ticket_price) AS avg_ticket_price
    FROM passenger_journey
""").display()

avg_ticket_price
14575.0


In [0]:
spark.sql("""
    SELECT to_city, COUNT(*) AS num_bookings
    FROM passenger_journey
    GROUP BY to_city
    ORDER BY num_bookings DESC
    LIMIT 1
""").display()

to_city,num_bookings
Delhi,5


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, sum as spark_sum, row_number

In [0]:
flight_revenue = (
    df_journey.groupBy("flight_id")
    .agg(spark_sum("ticket_price").alias("flight_revenue"))
)

window_top_flights = Window.orderBy(col("flight_revenue").desc())

top3_flights = (
    flight_revenue
    .withColumn("rank", row_number().over(window_top_flights))
    .filter(col("rank") <= 3)
)

top3_flights.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


flight_id,flight_revenue,rank
F101,39000,1
F103,38000,2
F109,28000,3


In [0]:
route_revenue_by_airline = (
    df_journey.groupBy("airline", "from_city", "to_city")
    .agg(spark_sum("ticket_price").alias("route_revenue"))
)

window_route_per_airline = Window.partitionBy("airline").orderBy(col("route_revenue").desc())

top_routes_by_airline = (
    route_revenue_by_airline
    .withColumn("rank_within_airline", row_number().over(window_route_per_airline))
    .filter(col("rank_within_airline") == 1)
)

top_routes_by_airline.display()

airline,from_city,to_city,route_revenue,rank_within_airline
Air India,Chennai,Bangalore,25000,1
Akasa,Delhi,Chennai,28000,1
Indigo,Hyderabad,Delhi,39000,1
Vistara,Bangalore,Hyderabad,38000,1


In [0]:
window_running = Window.orderBy("booking_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_revenue = (
    df_journey
    .withColumn("running_revenue", spark_sum("ticket_price").over(window_running))
    .select("booking_date", "ticket_price", "running_revenue")
    .orderBy("booking_date")
)

running_revenue.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


booking_date,ticket_price,running_revenue
2026-06-01,8500,8500
2026-06-01,22000,30500
2026-06-02,9000,39500
2026-06-02,15000,54500
2026-06-03,7500,62000
2026-06-03,25000,87000
2026-06-04,10000,97000
2026-06-04,17000,114000
2026-06-05,9500,123500
2026-06-05,28000,151500


In [0]:
airline_revenue = (
    df_journey.groupBy("airline")
    .agg(spark_sum("ticket_price").alias("total_revenue"))
)

window_rank_airlines = Window.orderBy(col("total_revenue").desc())

ranked_airlines = airline_revenue.withColumn("rank", rank().over(window_rank_airlines))

ranked_airlines.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


airline,total_revenue,rank
Indigo,90000,1
Vistara,71500,2
Air India,68000,3
Akasa,62000,4


In [0]:
destination_popularity = (
    df_journey.groupBy("to_city")
    .count()
    .withColumnRenamed("count", "num_bookings")
)

window_dense_rank_dest = Window.orderBy(col("num_bookings").desc())

dense_ranked_destinations = (
    destination_popularity
    .withColumn("dense_rank", dense_rank().over(window_dense_rank_dest))
)

dense_ranked_destinations.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


to_city,num_bookings,dense_rank
Delhi,5,1
Hyderabad,4,2
Mumbai,3,3
Chennai,2,4
Kolkata,2,4
Bangalore,2,4
Pune,1,5
Goa,1,5


In [0]:
delta_path_method1 = "/Volumes/mini_cap/default/airline_data/delta/booking_master_save"

df_journey.write.format("delta").mode("overwrite").save(delta_path_method1)

In [0]:
df_journey.write.format("delta").mode("overwrite").saveAsTable("mini_cap.default.booking_master")

In [0]:
%sql
CREATE OR REPLACE TABLE mini_cap.default.booking_master_sql
USING DELTA
AS
SELECT * FROM mini_cap.default.booking_master

num_affected_rows,num_inserted_rows


In [0]:
print("Day 1 (initial load) row count in booking_master:")
spark.sql("SELECT COUNT(*) AS day1_count FROM mini_cap.default.booking_master").show()

Day 1 (initial load) row count in booking_master:
+----------+
|day1_count|
+----------+
|        20|
+----------+



In [0]:
from pyspark.sql import Row
existing_ids = [row["booking_id"] for row in df_journey.select("booking_id").limit(5).collect()]
price_updates = spark.createDataFrame([
    Row(booking_id=existing_ids[0], ticket_price=27000),
    Row(booking_id=existing_ids[1], ticket_price=9500),
    Row(booking_id=existing_ids[2], ticket_price=15500),
    Row(booking_id=existing_ids[3], ticket_price=31000),
    Row(booking_id=existing_ids[4], ticket_price=7200),
])

display(price_updates)

booking_id,ticket_price
B1001,27000
B1002,9500
B1003,15500
B1004,31000
B1005,7200


In [0]:
import random

sample_flights = df_flights_transformed.limit(5).collect()  # reuse a few real flights

new_bookings_data = []
for i in range(10):
    f = sample_flights[i % len(sample_flights)]
    new_bookings_data.append(Row(
        booking_id=f"B2{100+i}",
        flight_id=f["flight_id"],
        passenger_name=f"New Passenger {i+1}",
        travel_class=random.choice(["Economy", "Business"]),
        ticket_price=random.choice([7000, 12000, 25000]),
        booking_date="2026-06-02"
    ))

df_new_bookings_raw = spark.createDataFrame(new_bookings_data)
display(df_new_bookings_raw)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date
B2100,F101,New Passenger 1,Economy,7000,2026-06-02
B2101,F102,New Passenger 2,Economy,25000,2026-06-02
B2102,F103,New Passenger 3,Economy,12000,2026-06-02
B2103,F104,New Passenger 4,Economy,7000,2026-06-02
B2104,F105,New Passenger 5,Business,12000,2026-06-02
B2105,F101,New Passenger 6,Economy,7000,2026-06-02
B2106,F102,New Passenger 7,Business,25000,2026-06-02
B2107,F103,New Passenger 8,Economy,7000,2026-06-02
B2108,F104,New Passenger 9,Economy,7000,2026-06-02
B2109,F105,New Passenger 10,Business,12000,2026-06-02


In [0]:
from pyspark.sql.functions import when, col

df_new_bookings = df_new_bookings_raw.withColumn(
    "price_band",
    when(col("ticket_price") > 20000, "Premium")
    .when(col("ticket_price") > 10000, "Standard")
    .otherwise("Budget")
)

display(df_new_bookings)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,price_band
B2100,F101,New Passenger 1,Economy,7000,2026-06-02,Budget
B2101,F102,New Passenger 2,Economy,25000,2026-06-02,Premium
B2102,F103,New Passenger 3,Economy,12000,2026-06-02,Standard
B2103,F104,New Passenger 4,Economy,7000,2026-06-02,Budget
B2104,F105,New Passenger 5,Business,12000,2026-06-02,Standard
B2105,F101,New Passenger 6,Economy,7000,2026-06-02,Budget
B2106,F102,New Passenger 7,Business,25000,2026-06-02,Premium
B2107,F103,New Passenger 8,Economy,7000,2026-06-02,Budget
B2108,F104,New Passenger 9,Economy,7000,2026-06-02,Budget
B2109,F105,New Passenger 10,Business,12000,2026-06-02,Standard


In [0]:
price_updates.createOrReplaceTempView("price_updates_view")
df_new_bookings.createOrReplaceTempView("new_bookings_view")

In [0]:
%sql
MERGE INTO mini_cap.default.booking_master AS target
USING price_updates_view AS source
ON target.booking_id = source.booking_id
WHEN MATCHED THEN
  UPDATE SET target.ticket_price = source.ticket_price,
             target.price_band = CASE
                WHEN source.ticket_price > 20000 THEN 'Premium'
                WHEN source.ticket_price > 10000 THEN 'Standard'
                ELSE 'Budget'
             END

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
MERGE INTO mini_cap.default.booking_master AS target
USING new_bookings_view AS source
ON target.booking_id = source.booking_id
WHEN NOT MATCHED THEN
  INSERT (booking_id, flight_id, passenger_name, travel_class, ticket_price, booking_date, price_band)
  VALUES (source.booking_id, source.flight_id, source.passenger_name, source.travel_class, source.ticket_price, source.booking_date, source.price_band)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,0,0,10


In [0]:
%sql
SELECT COUNT(*) AS day2_count FROM mini_cap.default.booking_master

day2_count
30


In [0]:
%sql
DESCRIBE HISTORY mini_cap.default.booking_master

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-06-19T10:41:38.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(booking_id#15432 = booking_id#14413)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1259750085640635),17f2a4af-ccee-4311-925a-2537eca4f87b,0619-095033-b8e21317-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 8, numTargetBytesAdded -> 29695, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1433, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 10, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1388)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-19T10:41:23.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1259750085640635),a794aada-43b7-4142-9fd9-39e9f4c4c248,0619-095033-b8e21317-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 9989, p25FileSize -> 5351, numDeletionVectorsRemoved -> 1, minFileSize -> 5351, numAddedFiles -> 1, maxFileSize -> 5351, p75FileSize -> 5351, p50FileSize -> 5351, numAddedBytes -> 5351)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-19T10:41:21.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(booking_id#14440 = booking_id#14392)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1259750085640635),a794aada-43b7-4142-9fd9-39e9f4c4c248,0619-095033-b8e21317-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4667, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 4261, materializeSourceTimeMs -> 180, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1967, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2030)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-19T10:35:59.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1259750085640635),88c93d16-8e98-4f1a-8e24-00ee0c3a1b20,0619-095033-b8e21317-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 5322)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("mini_cap.default.booking_master")
print("Version 0 row count:", df_v0.count())
display(df_v0)

Version 0 row count: 20


passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat,extra_baggage
Rahul Sharma,F101,B1001,Economy,8500,2026-06-01,Budget,Indigo,Hyderabad,Delhi,140,On Time,No,Veg,Window,true
Priya Reddy,F101,B1002,Business,22000,2026-06-01,Premium,Indigo,Hyderabad,Delhi,140,On Time,No,Non-Veg,Aisle,false
Amit Kumar,F102,B1003,Economy,9000,2026-06-02,Budget,Air India,Mumbai,Chennai,120,Delayed,Yes,Veg,Middle,false
Sneha Patel,F103,B1004,Premium Economy,15000,2026-06-02,Standard,Vistara,Bangalore,Hyderabad,90,On Time,No,Jain,Window,true
Farhan Ali,F104,B1005,Economy,7500,2026-06-03,Budget,Indigo,Delhi,Mumbai,130,Cancelled,No,Non-Veg,Aisle,false
Neha Singh,F105,B1006,Business,25000,2026-06-03,Premium,Air India,Chennai,Bangalore,80,On Time,No,Veg,Window,true
Arjun Verma,F106,B1007,Economy,10000,2026-06-04,Budget,Akasa,Pune,Delhi,150,Delayed,Yes,Veg,Middle,false
Meera Nair,F107,B1008,Premium Economy,17000,2026-06-04,Standard,Vistara,Hyderabad,Kolkata,160,On Time,No,Jain,Window,true
Kiran Rao,F108,B1009,Economy,9500,2026-06-05,Budget,Indigo,Mumbai,Hyderabad,110,On Time,No,Veg,Aisle,false
Nisha Reddy,F109,B1010,Business,28000,2026-06-05,Premium,Akasa,Delhi,Chennai,145,Delayed,Yes,Non-Veg,Window,true


In [0]:
df_v1 = spark.read.format("delta").option("versionAsOf", 1).table("mini_cap.default.booking_master")
print("Version 1 row count:", df_v1.count())
display(df_v1)

Version 1 row count: 20


passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat,extra_baggage
Neha Singh,F105,B1006,Business,25000,2026-06-03,Premium,Air India,Chennai,Bangalore,80,On Time,No,Veg,Window,true
Arjun Verma,F106,B1007,Economy,10000,2026-06-04,Budget,Akasa,Pune,Delhi,150,Delayed,Yes,Veg,Middle,false
Meera Nair,F107,B1008,Premium Economy,17000,2026-06-04,Standard,Vistara,Hyderabad,Kolkata,160,On Time,No,Jain,Window,true
Kiran Rao,F108,B1009,Economy,9500,2026-06-05,Budget,Indigo,Mumbai,Hyderabad,110,On Time,No,Veg,Aisle,false
Nisha Reddy,F109,B1010,Business,28000,2026-06-05,Premium,Akasa,Delhi,Chennai,145,Delayed,Yes,Non-Veg,Window,true
David Thomas,F110,B1011,Economy,8000,2026-06-06,Budget,Air India,Bangalore,Mumbai,95,On Time,No,Veg,Middle,false
Ayesha Khan,F111,B1012,Premium Economy,16000,2026-06-06,Standard,Indigo,Hyderabad,Goa,75,On Time,No,Jain,Window,true
Rohit Sharma,F112,B1013,Economy,7000,2026-06-07,Budget,Vistara,Goa,Delhi,150,Cancelled,No,Veg,Aisle,false
Pooja Mehta,F113,B1014,Business,24000,2026-06-07,Premium,Akasa,Chennai,Pune,100,On Time,No,Non-Veg,Window,true
Sanjay Gupta,F114,B1015,Economy,10500,2026-06-08,Standard,Air India,Kolkata,Bangalore,170,Delayed,Yes,Veg,Middle,false


In [0]:
df_latest = spark.read.format("delta").table("mini_cap.default.booking_master")
print("Latest version row count:", df_latest.count())
display(df_latest)

Latest version row count: 30


passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat,extra_baggage
Neha Singh,F105,B1006,Business,25000,2026-06-03,Premium,Air India,Chennai,Bangalore,80,On Time,No,Veg,Window,true
Arjun Verma,F106,B1007,Economy,10000,2026-06-04,Budget,Akasa,Pune,Delhi,150,Delayed,Yes,Veg,Middle,false
Meera Nair,F107,B1008,Premium Economy,17000,2026-06-04,Standard,Vistara,Hyderabad,Kolkata,160,On Time,No,Jain,Window,true
Kiran Rao,F108,B1009,Economy,9500,2026-06-05,Budget,Indigo,Mumbai,Hyderabad,110,On Time,No,Veg,Aisle,false
Nisha Reddy,F109,B1010,Business,28000,2026-06-05,Premium,Akasa,Delhi,Chennai,145,Delayed,Yes,Non-Veg,Window,true
David Thomas,F110,B1011,Economy,8000,2026-06-06,Budget,Air India,Bangalore,Mumbai,95,On Time,No,Veg,Middle,false
Ayesha Khan,F111,B1012,Premium Economy,16000,2026-06-06,Standard,Indigo,Hyderabad,Goa,75,On Time,No,Jain,Window,true
Rohit Sharma,F112,B1013,Economy,7000,2026-06-07,Budget,Vistara,Goa,Delhi,150,Cancelled,No,Veg,Aisle,false
Pooja Mehta,F113,B1014,Business,24000,2026-06-07,Premium,Akasa,Chennai,Pune,100,On Time,No,Non-Veg,Window,true
Sanjay Gupta,F114,B1015,Economy,10500,2026-06-08,Standard,Air India,Kolkata,Bangalore,170,Delayed,Yes,Veg,Middle,false


In [0]:
sample_id = existing_ids[0] 

print("Before merge (Version 0):")
df_v0.filter(col("booking_id") == sample_id).select("booking_id", "ticket_price", "price_band").show()

print("After merge (Latest):")
df_latest.filter(col("booking_id") == sample_id).select("booking_id", "ticket_price", "price_band").show()

Before merge (Version 0):
+----------+------------+----------+
|booking_id|ticket_price|price_band|
+----------+------------+----------+
|     B1001|        8500|    Budget|
+----------+------------+----------+

After merge (Latest):
+----------+------------+----------+
|booking_id|ticket_price|price_band|
+----------+------------+----------+
|     B1001|       27000|   Premium|
+----------+------------+----------+



In [0]:
%sql
OPTIMIZE mini_cap.default.booking_master

path,metrics
abfss://unity-catalog-storage@dbstoragexilgy5hegv2j6.dfs.core.windows.net/7405616321077284/__unitystorage/catalogs/ed721532-9b4b-43fa-9216-b3def1f17e08/tables/eb7cde1a-fbdb-4577-a8f5-66ed45815f6e,"List(1, 9, List(5513, 5513, 5513.0, 1, 5513), List(3691, 5351, 3894.0, 9, 35046), 0, null, null, 0, 1, 9, 0, true, 0, 0, 1781865960039, 1781865961928, 8, 1, null, List(0, 0), null, 16, 16, 502, 0, null, null)"


In [0]:
%sql
OPTIMIZE mini_cap.default.booking_master
ZORDER BY (flight_id)

path,metrics
abfss://unity-catalog-storage@dbstoragexilgy5hegv2j6.dfs.core.windows.net/7405616321077284/__unitystorage/catalogs/ed721532-9b4b-43fa-9216-b3def1f17e08/tables/eb7cde1a-fbdb-4577-a8f5-66ed45815f6e,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 5513), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781865983508, 1781865984134, 8, 0, null, List(0, 0), null, 16, 16, 0, 0, null, null)"


In [0]:
%sql
VACUUM mini_cap.default.booking_master

path
abfss://unity-catalog-storage@dbstoragexilgy5hegv2j6.dfs.core.windows.net/7405616321077284/__unitystorage/catalogs/ed721532-9b4b-43fa-9216-b3def1f17e08/tables/eb7cde1a-fbdb-4577-a8f5-66ed45815f6e


In [0]:
%sql
DESCRIBE HISTORY mini_cap.default.booking_master

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-06-19T10:46:46.001Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(1259750085640635),007ffbba-a5f9-45a0-9f86-603f0b53ea51,0619-095033-b8e21317-v2n,5,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
5,2026-06-19T10:46:46.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(1259750085640635),007ffbba-a5f9-45a0-9f86-603f0b53ea51,0619-095033-b8e21317-v2n,4,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
4,2026-06-19T10:46:01.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1259750085640635),6d59631a-951a-49ec-aff5-21185983d067,0619-095033-b8e21317-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 9, numRemovedBytes -> 35046, p25FileSize -> 5513, numDeletionVectorsRemoved -> 0, minFileSize -> 5513, numAddedFiles -> 1, maxFileSize -> 5513, p75FileSize -> 5513, p50FileSize -> 5513, numAddedBytes -> 5513)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-19T10:41:38.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(booking_id#15432 = booking_id#14413)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1259750085640635),17f2a4af-ccee-4311-925a-2537eca4f87b,0619-095033-b8e21317-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 8, numTargetBytesAdded -> 29695, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1433, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 10, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1388)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-19T10:41:23.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1259750085640635),a794aada-43b7-4142-9fd9-39e9f4c4c248,0619-095033-b8e21317-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 9989, p25FileSize -> 5351, numDeletionVectorsRemoved -> 1, minFileSize -> 5351, numAddedFiles -> 1, maxFileSize -> 5351, p75FileSize -> 5351, p50FileSize -> 5351, numAddedBytes -> 5351)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-19T10:41:21.000Z,144600303012599,azuser7231_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(booking_id#14440 = booking_id#14392)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1259750085640635),a794aada-43b7-4142-9fd9-39e9f4c4c248,0619-095033-b8e21317-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4667, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 4261, materializeSourceTimeMs -> 180, numTarge

In [0]:
%sql
DESCRIBE TABLE EXTENDED mini_cap.default.booking_master

col_name,data_type,comment
passenger_name,string,null
flight_id,string,null
booking_id,string,null
travel_class,string,null
ticket_price,int,null
booking_date,date,null
price_band,string,null
airline,string,null
from_city,string,null
to_city,string,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS mini_cap.default.booking_master_external
USING DELTA
AS
SELECT * FROM mini_cap.default.booking_master

num_affected_rows,num_inserted_rows


In [0]:
df_journey.createOrReplaceTempView("journey_temp_view")
print("Temp view created — visible only in this notebook session")

Temp view created — visible only in this notebook session


In [0]:
df_journey.createOrReplaceGlobalTempView("journey_global_temp_view")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5381388101595026>, line 1
----> 1 df_journey.createOrReplaceGlobalTempView("journey_global_temp_view")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2129, in DataFrame.createOrReplaceGlobalTempView(self, name)
   2125 def createOrReplaceGlobalTempView(self, name: str) -> None:
   2126     command = plan.CreateView(
   2127         child=self._plan, name=name, is_global=True, replace=True
   2128     ).command(session=self._session.client)
-> 2129     _, _, ei = self._session.client.execute_command(command, self._plan.observations)
   2130     self._execution_info = ei

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.use

In [0]:
spark.sql("SELECT * FROM global_temp.journey_global_temp_view LIMIT 5").display()